In [13]:
set.seed(123)

###############################
# COMMON FUNCTIONS
###############################

sigmoid <- function(z) {
  1 / (1 + exp(-z))
}

compute_gradient <- function(X, y, beta) {
  p <- sigmoid(X %*% beta)
  t(X) %*% (y - p)
}

compute_hessian <- function(X, beta) {
  p <- sigmoid(X %*% beta)
  W <- diag(as.vector(p * (1 - p)))
  -t(X) %*% W %*% X
}

compute_error <- function(beta_est, true_beta) {
  sqrt(sum((beta_est - true_beta)^2))
}

split_data <- function(X, y, k = 4) {
  n <- nrow(X)
  idx <- split(1:n, rep(1:k, length.out = n))

  lapply(idx, function(i) {
    list(X = X[i, , drop = FALSE], y = y[i])
  })
}

true_beta <- matrix(c(0.5, 1), ncol = 1)

sample_sizes <- c(100, 200, 400, 1000, 2000)

all_results <- list()

In [14]:
###############################
# ALGORITHM 1: VECTORIZED NEWTON-RAPHSON
###############################

newton_vectorized <- function(nodes, tol = 1e-6, max_iter = 100) {

  beta <- matrix(0, ncol = 1, nrow = 2)
  history <- list()
  start_total <- Sys.time()

  for (iter in 1:max_iter) {

    iter_start <- Sys.time()

    gradients <- lapply(nodes, function(node) {
      compute_gradient(node$X, node$y, beta)
    })

    hessians <- lapply(nodes, function(node) {
      compute_hessian(node$X, beta)
    })

    g <- Reduce("+", gradients)
    H <- Reduce("+", hessians)

    ridge <- diag(1e-8, nrow(H))
    beta_new <- beta - solve(H + ridge, g)

    iter_time <- as.numeric(Sys.time() - iter_start, units = "secs")

    history[[iter]] <- data.frame(
      Method = "Vectorized",
      Iteration = iter,
      Beta0 = beta_new[1, 1],
      Beta1 = beta_new[2, 1],
      Error = compute_error(beta_new, true_beta),
      IterTime = iter_time
    )

    if (sqrt(sum((beta_new - beta)^2)) < tol) {
      beta <- beta_new
      break
    }

    beta <- beta_new
  }

  list(
    beta = beta,
    iterations = iter,
    runtime = as.numeric(Sys.time() - start_total, units = "secs"),
    history = do.call(rbind, history)
  )
}

In [15]:
###############################
# ALGORITHM 2: PARALLEL NEWTON-RAPHSON
###############################

newton_parallel <- function(nodes, tol = 1e-6, max_iter = 100) {

  library(parallel)

  beta <- matrix(0, ncol = 1, nrow = 2)
  history <- list()

  cl <- makeCluster(4)
  clusterExport(
    cl,
    c("compute_gradient", "compute_hessian", "sigmoid"),
    envir = environment()
  )

  start_total <- Sys.time()

  for (iter in 1:max_iter) {

    iter_start <- Sys.time()

    gradients <- parLapply(cl, nodes, function(node, beta) {
      compute_gradient(node$X, node$y, beta)
    }, beta)

    hessians <- parLapply(cl, nodes, function(node, beta) {
      compute_hessian(node$X, beta)
    }, beta)

    g <- Reduce("+", gradients)
    H <- Reduce("+", hessians)

    ridge <- diag(1e-8, nrow(H))
    beta_new <- beta - solve(H + ridge, g)

    iter_time <- as.numeric(Sys.time() - iter_start, units = "secs")

    history[[iter]] <- data.frame(
      Method = "Parallel",
      Iteration = iter,
      Beta0 = beta_new[1, 1],
      Beta1 = beta_new[2, 1],
      Error = compute_error(beta_new, true_beta),
      IterTime = iter_time
    )

    if (sqrt(sum((beta_new - beta)^2)) < tol) {
      beta <- beta_new
      break
    }

    beta <- beta_new
  }

  stopCluster(cl)

  list(
    beta = beta,
    iterations = iter,
    runtime = as.numeric(Sys.time() - start_total, units = "secs"),
    history = do.call(rbind, history)
  )
}

In [16]:
###############################
# ALGORITHM 3: INCREMENTAL NEWTON-RAPHSON
###############################

newton_incremental <- function(nodes, tol = 1e-6, max_iter = 100) {

  beta <- matrix(0, ncol = 1, nrow = 2)
  history <- list()

  local_gradients <- lapply(nodes, function(node) {
    compute_gradient(node$X, node$y, beta)
  })

  local_hessians <- lapply(nodes, function(node) {
    compute_hessian(node$X, beta)
  })

  g_global <- Reduce("+", local_gradients)
  H_global <- Reduce("+", local_hessians)

  start_total <- Sys.time()

  for (iter in 1:max_iter) {

    iter_start <- Sys.time()

    for (i in 1:length(nodes)) {

      g_new <- compute_gradient(nodes[[i]]$X, nodes[[i]]$y, beta)
      H_new <- compute_hessian(nodes[[i]]$X, beta)

      g_global <- g_global - local_gradients[[i]] + g_new
      H_global <- H_global - local_hessians[[i]] + H_new

      local_gradients[[i]] <- g_new
      local_hessians[[i]] <- H_new
    }

    g <- g_global
    H <- H_global

    ridge <- diag(1e-8, nrow(H))
    beta_new <- beta - solve(H + ridge, g)

    iter_time <- as.numeric(Sys.time() - iter_start, units = "secs")

    history[[iter]] <- data.frame(
      Method = "Incremental",
      Iteration = iter,
      Beta0 = beta_new[1, 1],
      Beta1 = beta_new[2, 1],
      Error = compute_error(beta_new, true_beta),
      IterTime = iter_time
    )

    if (sqrt(sum((beta_new - beta)^2)) < tol) {
      beta <- beta_new
      break
    }

    beta <- beta_new
  }

  list(
    beta = beta,
    iterations = iter,
    runtime = as.numeric(Sys.time() - start_total, units = "secs"),
    history = do.call(rbind, history)
  )
}

In [17]:
###############################
# MAIN EXPERIMENT LOOP
###############################

for (n in sample_sizes) {

  cat("\n=============================\n")
  cat("Experiment n =", n, "\n")
  cat("=============================\n")

  x <- rnorm(n)
  z <- 0.5 + x
  p <- sigmoid(z)
  y <- rbinom(n, 1, p)
  X <- cbind(1, x)

  nodes <- split_data(X, y, 4)

  ###############################
  # SELECT ALGORITHM TO RUN
  ###############################

  # Uncomment the one you want to use.
  # Comment the others.

  res <- newton_vectorized(nodes)
  res <- newton_parallel(nodes)
  res <- newton_incremental(nodes)

  hist <- res$history
  hist$SampleSize <- n
  rownames(hist) <- NULL

  print(hist)

  summary <- data.frame(
    SampleSize = n,
    Method = hist$Method[1],
    Beta0 = res$beta[1, 1],
    Beta1 = res$beta[2, 1],
    Iterations = res$iterations,
    Runtime = res$runtime,
    Error = compute_error(res$beta, true_beta)
  )

  all_results[[as.character(n)]] <- summary

  cat("\n--- Summary Table ---\n")
  print(summary)
}

###############################
# FINAL COMPARISON
###############################

final_results <- do.call(rbind, all_results)
rownames(final_results) <- NULL

cat("\n===== FINAL RESULTS =====\n")
print(final_results)

cat("\n===== SORTED BY RUNTIME =====\n")
print(final_results[order(final_results$Runtime), ])

cat("\n===== SORTED BY ERROR =====\n")
print(final_results[order(final_results$Error), ])


Experiment n = 100 
       Method Iteration     Beta0     Beta1     Error     IterTime SampleSize
1 Incremental         1 0.5205076 0.8792828 0.1224468 0.0002031326        100
2 Incremental         2 0.6445713 1.1529921 0.2104934 0.0002484322        100
3 Incremental         3 0.6632977 1.1961655 0.2552392 0.0002195835        100
4 Incremental         4 0.6636970 1.1970972 0.2562109 0.0002243519        100
5 Incremental         5 0.6636972 1.1970976 0.2562113 0.0002055168        100

--- Summary Table ---
  SampleSize      Method     Beta0    Beta1 Iterations     Runtime     Error
x        100 Incremental 0.6636972 1.197098          5 0.004936934 0.2562113

Experiment n = 200 
       Method Iteration     Beta0     Beta1     Error     IterTime SampleSize
1 Incremental         1 0.6512044 0.6926484 0.3425314 0.0002369881        200
2 Incremental         2 0.7832834 0.9177818 0.2949734 0.0002646446        200
3 Incremental         3 0.8018635 0.9492647 0.3060975 0.0002450943        200
4